In [20]:
import os
import pickle
from tqdm import tqdm
import torch
from token_utils_rep import EHRTokenizer
from dataset_utils_rep import HBERTPretrainEHRDataset, batcher
from torch.utils.data import DataLoader
from HEART_rep import HBERT_Pretrain
from set_seed_utils import set_random_seed

In [21]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


In [22]:
args = {
    "seed": 0,
    "dataset": "MIMIC-III",  # MIMIC-III, MIMIC-IV
    "batch_size": 32,
    "lr": 2e-5,
    "epochs": 20,
    "encoder": "hi",
    "mask_rate": 0.7,
    "anomaly_loss_weight": 1,
    "num_hidden_layers": 5,
    "num_attention_heads": 6,
    "attention_probs_dropout_prob": 0.2,
    "hidden_dropout_prob": 0.2,
    "edge_hidden_size": 32,
    "hidden_size": 288,  # must be divisible by num_attention_heads
    "intermediate_size": 288,
    "gnn_n_heads": 1,
    "gnn_temp": 1,
    "gat": "None",  # dotattn, None
    "diag_med_emb": "simple",  # simple, tree
}

In [23]:
# here we only use the MIMIC dataset
args['max_visit_size'] = 15
args['predicted_token_type'] = ["diag", "lab", "pro"]
args['mask_token_id'] = {"diag":3, "lab":4, "pro":5}
args['special_tokens'] = ("[PAD]", "[CLS]", "[SEP]", 
                       "[MASK0]", "[MASK1]", "[MASK2]", "[MASK3]")
# note that here "[MASK0]", "[MASK1]", "[MASK2]", "[MASK3]" are used for masking pretraining task
# codes that are actually masked are not in the input sequence

In [24]:
full_data_path = f"/home/lideyi/HeteroGT-cuda/data_process/{args['dataset']}-processed/mimic.pkl"
pretrain_data_path = f"/home/lideyi/HeteroGT-cuda/data_process/{args['dataset']}-processed/mimic_pretrain.pkl" # for pretraining

In [25]:
ehr_data = pickle.load(open(full_data_path, 'rb'))

In [26]:
diag_sentences = ehr_data["ICD9_CODE"].values.tolist()
lab_sentences = ehr_data["LAB_TEST"].values.tolist()
pro_sentences = ehr_data["PRO_CODE"].values.tolist()
gender_set = [["M"], ["F"]]
age_set = [[c] for c in set(ehr_data["AGE"].values.tolist())]
age_gender_set = [[str(c) + "_" + gender] \
    for c in set(ehr_data["AGE"].values.tolist()) for gender in ["M", "F"]]

In [27]:
# tokenizer used full data
tokenizer = EHRTokenizer(diag_sentences, lab_sentences, pro_sentences, gender_set, 
                         age_set, age_gender_set, special_tokens=args["special_tokens"])

In [28]:
ehr_pretrain_data = pickle.load(open(pretrain_data_path, 'rb'))
pretrain_dataset = HBERTPretrainEHRDataset(ehr_pretrain_data, tokenizer, 
                                  token_type=args['predicted_token_type'], 
                                  mask_rate=args['mask_rate'])

In [29]:
pretrain_dataloader = DataLoader(pretrain_dataset, batch_size=args["batch_size"], 
                                 collate_fn=batcher(pad_id = tokenizer.vocab.word2id["[PAD]"], 
                                                    n_token_type=len(args["predicted_token_type"]), is_train = True),
                                 shuffle=True)

In [30]:
batch = next(iter(pretrain_dataloader))
input_ids, input_types, edge_index, visit_positions, labels = batch

In [31]:
set_random_seed(args["seed"])

[INFO] Random seed set to 0


In [32]:
exp_name = "Pretrain-ExBEHRT" \
    + "-" + str(args["dataset"]) \
    + "-" + str(args["encoder"]) \
    + "-" + str(args["mask_rate"]) \
    + "-" + str(args["hidden_size"]) \
    + "-" + str(args["edge_hidden_size"]) \
    + "-" + str(args["num_hidden_layers"]) \
    + "-" + str(args["num_attention_heads"]) \
    + "-" + str(args["attention_probs_dropout_prob"]) \
    + "-" + str(args["hidden_dropout_prob"]) \
    + "-" + str(args["intermediate_size"]) \
    + "-" + str(args["gat"]) \
    + "-" + str(args["gnn_n_heads"]) \
    + "-" + str(args["gnn_temp"]) \
    + "-" + str(args["diag_med_emb"])
print(exp_name)

save_path = "./pretrained_models/" + exp_name
if not os.path.exists(save_path):
    os.makedirs(save_path)

Pretrain-ExBEHRT-MIMIC-III-hi-0.7-288-32-5-6-0.2-0.2-288-None-1-1-simple


In [33]:
args["vocab_size"] = len(args["special_tokens"]) + len(tokenizer.diag_voc.id2word) + \
                + len(tokenizer.lab_voc.id2word) + \
                + len(tokenizer.pro_voc.id2word) + \
                len(tokenizer.age_voc.id2word) + \
                len(tokenizer.gender_voc.id2word) + \
                len(tokenizer.age_gender_voc.id2word)

args["label_vocab_size"] = {"diag":len(tokenizer.diag_voc.id2word), 
                            "lab":len(tokenizer.lab_voc.id2word), 
                            "pro":len(tokenizer.pro_voc.id2word)}  # {token_type: vocab_size}

In [34]:
loss_entity = ["diag", "lab", "pro"]

In [35]:
model = HBERT_Pretrain(args, tokenizer).to(device)

In [36]:
optimizer = torch.optim.AdamW(model.parameters(), lr=args["lr"])

In [37]:
for epoch in range(1, 1 + args["epochs"]):
    train_iter = tqdm(pretrain_dataloader, ncols=140)
    model.train()
    ave_loss, ave_loss_dict = 0., {token_type: 0. for token_type in loss_entity}

    for step, batch in enumerate(train_iter):

        batch = [x.to(device) if isinstance(x, torch.Tensor) else x for x in batch]
        loss, loss_dict, perf_dict = model(*batch)

        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        train_iter.set_description(f"Epoch:{epoch: 03d}, Step:{step: 03d}, loss:{loss.item():.4f}, diag:{loss_dict['diag']:.4f}")

        ave_loss += loss.item()
        ave_loss_dict = {token_type: ave_loss_dict[token_type] + loss_dict[token_type] for token_type in loss_entity}

    ave_loss /= (step + 1)
    ave_loss_dict = {token_type: ave_loss_dict[token_type] / (step + 1) for token_type in loss_entity}
    print(f"Epoch {epoch} finished, ave_loss: {ave_loss:.4f}, ave_loss_dict: {ave_loss_dict}, perf_dict: {perf_dict}")

Epoch: 01, Step: 723, loss:0.0303, diag:0.0187: 100%|█████████████████████████████████████████████████████| 724/724 [00:23<00:00, 30.78it/s]


Epoch 1 finished, ave_loss: 0.1237, ave_loss_dict: {'diag': 0.11189460412480802, 'lab': 0.15244596088768203, 'pro': 0.10680162958872828}, perf_dict: {'diag': {'precision': 0.0, 'recall': 0.0, 'f1': 0.0}, 'lab': {'precision': 0.0, 'recall': 0.0, 'f1': 0.0}, 'pro': {'precision': 0.0, 'recall': 0.0, 'f1': 0.0}}


Epoch: 02, Step: 723, loss:0.0251, diag:0.0099: 100%|█████████████████████████████████████████████████████| 724/724 [00:22<00:00, 32.46it/s]


Epoch 2 finished, ave_loss: 0.0305, ave_loss_dict: {'diag': 0.015270914034712134, 'lab': 0.06059133722987129, 'pro': 0.015566064057696755}, perf_dict: {'diag': {'precision': 0.0, 'recall': 0.0, 'f1': 0.0}, 'lab': {'precision': 0.0, 'recall': 0.0, 'f1': 0.0}, 'pro': {'precision': 0.0, 'recall': 0.0, 'f1': 0.0}}


Epoch: 03, Step: 723, loss:0.0294, diag:0.0105: 100%|█████████████████████████████████████████████████████| 724/724 [00:22<00:00, 32.37it/s]


...

Epoch 18 finished, ave_loss: 0.0269, ave_loss_dict: {'diag': 0.013606980426278151, 'lab': 0.054849636797582244, 'pro': 0.012198737716991762}, perf_dict: {'diag': {'precision': 0.0, 'recall': 0.0, 'f1': 0.0}, 'lab': {'precision': 0.002, 'recall': 0.0013333333333333333, 'f1': 0.0015555555555555553}, 'pro': {'precision': 0.004369538077403246, 'recall': 0.003329171868497711, 'f1': 0.003703703703703704}}


Epoch: 19, Step: 723, loss:0.0265, diag:0.0117: 100%|█████████████████████████████████████████████████████| 724/724 [00:24<00:00, 29.08it/s]


Epoch 19 finished, ave_loss: 0.0268, ave_loss_dict: {'diag': 0.013562349008283068, 'lab': 0.054692061890678184, 'pro': 0.012100973210261507}, perf_dict: {'diag': {'precision': 0.0005005005005005005, 'recall': 0.0005005005005005005, 'f1': 0.0005005005005005005}, 'lab': {'precision': 0.003, 'recall': 0.003, 'f1': 0.0027555555555555554}, 'pro': {'precision': 0.0024968789013732834, 'recall': 0.0024968789013732834, 'f1': 0.0024968789013732834}}


Epoch: 20, Step: 723, loss:0.0286, diag:0.0121: 100%|█████████████████████████████████████████████████████| 724/724 [00:24<00:00, 29.19it/s]

Epoch 20 finished, ave_loss: 0.0267, ave_loss_dict: {'diag': 0.013517903230919693, 'lab': 0.054536589776366454, 'pro': 0.012014852104790879}, perf_dict: {'diag': {'precision': 0.0, 'recall': 0.0, 'f1': 0.0}, 'lab': {'precision': 0.0006666666666666666, 'recall': 0.0006666666666666666, 'f1': 0.0006666666666666666}, 'pro': {'precision': 0.0024968789013732834, 'recall': 0.0012484394506866417, 'f1': 0.0016229712858926342}}


In [38]:
torch.save(model.cpu().state_dict(), f"{save_path}/pretrained_model.pt")